# AFR / EUR studies: sketch dimension, identity LD and SER fallback (data processing)

From `28.FigS7.AFR_EUR_Cross_Reference` `[cross_ref_B_sweep]` and `[cross_ref_ld_similarity]`. Three questions:
1. Does the `R_finite` setting (panel size, 800, 10,000) change how often the SER fallback is triggered with matched and cross-population LD?
2. Is AFR LD closer to an identity matrix than EUR LD?
3. Does identity LD trigger the SER fallback?
4. Power and FDR of the SuSiE-rRSS pipeline result with each LD and `R_finite`.

**Outputs:**
- `figure_data/FigS7.AFR_EUR_B_Identity_Check_Fallback.rds` (+ `.csv`) — SER fallback per study × LD × `R_finite`, pooled and per NCP, with median `Q_art` and `D_CS,max`
- `figure_data/FigS7.AFR_EUR_B_Identity_Check_Power_FDR.rds` (+ `.csv`) — power and FDR (bootstrap 95% CI) per study × LD × `R_finite`, pooled and per NCP
- `figure_data/FigS7.AFR_EUR_B_Identity_Check_LD_Similarity.rds` (+ `.csv`) — per region: MSE, max |ΔLD| and correlation of the other population's LD and of identity against the study LD

In [10]:
source("../../R/utils.R")

data_name <- "FigS7.AFR_EUR_B_Identity_Check"
root      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_cross_ref_Bsweep"

## SER fallback by LD and `R_finite`

In [11]:
sweep <- list.files(file.path(root, "result"), pattern = "\\.rds$", recursive = TRUE, full.names = TRUE) %>%
  map_dfr(readRDS)
count(sweep, study, ld, B_setting, fit_failed)              # replicates per setting; failed fits are dropped below

## "Panel size" or the fixed R_finite value (any value passed as B_fixed)
B_lab  <- function(b) if_else(b == "panel", "Panel size", scales::comma(suppressWarnings(as.numeric(b))))
LD_lab <- c("Matched", "Cross-population", "Identity")

fb_rows <- sweep %>%
  filter(!fit_failed) %>%
  mutate(ncp       = round(n * h2g),
         LD        = factor(case_when(ld == "Identity" ~ "Identity", ld == study ~ "Matched", TRUE ~ "Cross-population"), levels = LD_lab),
         B_setting = fct_reorder(B_lab(B_setting), if_else(B_setting == "panel", 0, suppressWarnings(as.numeric(B_setting)))))

summarise_fb <- function(d) d %>%
  summarise(n = n(), n_fallback = sum(R_reliability_flag), fallback = n_fallback / n,
            Q_art_median = median(Q_art, na.rm = TRUE), D_CS_max_median = median(D_CS_max, na.rm = TRUE),
            panel_B = median(B), .groups = "drop")

fallback <- bind_rows(
  fb_rows %>% group_by(study, LD, B_setting) %>% summarise_fb() %>% mutate(ncp = NA_real_),
  fb_rows %>% group_by(study, LD, B_setting, ncp) %>% summarise_fb())

## pooled over NCP and number of causal variants
fallback %>% filter(is.na(ncp)) %>%
  transmute(study, LD, B_setting, B = panel_B, fallback = sprintf("%d/%d (%.1f%%)", n_fallback, n, 100 * fallback),
            Q_art_median, D_CS_max_median) %>% as.data.frame()

save_data(fallback, paste0(data_name, "_Fallback"))

study,ld,B_setting,fit_failed,n
<chr>,<chr>,<chr>,<lgl>,<int>
AFR,AFR,10000,FALSE,2400
AFR,AFR,893,FALSE,2400
AFR,AFR,panel,FALSE,2400
AFR,EUR,10000,FALSE,2400
AFR,EUR,893,FALSE,2400
AFR,EUR,panel,FALSE,2400
AFR,Identity,10000,FALSE,2400
AFR,Identity,893,FALSE,2400
EUR,AFR,10000,FALSE,2400


study,LD,B_setting,B,fallback,Q_art_median,D_CS_max_median
<chr>,<fct>,<fct>,<dbl>,<chr>,<dbl>,<dbl>
AFR,Matched,Panel size,893,53/2400 (2.2%),0.004079431,0.38702643
AFR,Matched,893,893,53/2400 (2.2%),0.004079431,0.38702643
AFR,Matched,"10,000",10000,1/2400 (0.0%),0.004099315,0.03918888
AFR,Cross-population,Panel size,1000,2267/2400 (94.5%),0.133824044,161.35569899
AFR,Cross-population,893,893,2268/2400 (94.5%),0.133923928,157.81066772
AFR,Cross-population,"10,000",10000,2260/2400 (94.2%),0.132216801,71.26504708
AFR,Identity,893,893,381/2400 (15.9%),0.000000000,5.84201179
AFR,Identity,"10,000",10000,304/2400 (12.7%),0.000000000,1.43670822
EUR,Matched,Panel size,1000,0/2400 (0.0%),0.006967129,0.48693858


## Power and FDR by LD and `R_finite`

Pipeline result per fit: the SER model when the reliability flag is set, otherwise the adjusted fit (as Fig S7b). The sweep keeps credible-set counts, not causal-variant counts, so power = credible sets holding a causal variant / causal variants (capped at the number of causal variants; it differs from Fig S7b only when one credible set holds two causal variants), and FDR = 1 − true / all credible sets. Bootstrap over replicates (`boot_cell_mix()` in `R/utils.R`).

In [12]:
pf_rows <- fb_rows %>%
  mutate(use_ser          = R_reliability_flag & !is.na(n_cs_ser),
         total_set_number = if_else(use_ser, n_cs_ser, n_cs),
         true_set_number  = if_else(use_ser, n_true_cs_ser, n_true_cs),
         total_var_number = n_causal,
         true_var_number  = pmin(true_set_number, n_causal))

boot_pf <- function(d, key) bind_cols(boot_cell_mix(d, 1000), tibble(panel_B = median(d$B)))
set.seed(1)
power_fdr <- bind_rows(
  pf_rows %>% group_by(study, LD, B_setting) %>% group_modify(boot_pf) %>% ungroup() %>% mutate(ncp = NA_real_),
  pf_rows %>% group_by(study, LD, B_setting, ncp) %>% group_modify(boot_pf) %>% ungroup()) %>%
  transmute(study, LD, B_setting, panel_B, ncp, R,
            Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
            FDR_est   = FDR,   FDR_lo   = FDR_lo,   FDR_hi   = FDR_hi) %>%
  pivot_longer(c(starts_with("Power_"), starts_with("FDR_")), names_to = c("metric", ".value"), names_sep = "_") %>%
  mutate(metric = factor(metric, levels = c("Power", "FDR")))

## pooled over NCP and number of causal variants
power_fdr %>% filter(is.na(ncp)) %>%
  transmute(study, LD, B_setting, B = panel_B, metric, value = sprintf("%.1f%% [%.1f, %.1f]", 100 * est, 100 * lo, 100 * hi)) %>%
  pivot_wider(names_from = metric, values_from = value) %>% as.data.frame()

save_data(power_fdr, paste0(data_name, "_Power_FDR"))

study,LD,B_setting,B,Power,FDR
<chr>,<fct>,<fct>,<dbl>,<chr>,<chr>
AFR,Matched,Panel size,893,"76.5% [75.0, 77.8]","0.5% [0.3, 0.8]"
AFR,Matched,893,893,"76.5% [75.1, 77.9]","0.5% [0.3, 0.8]"
AFR,Matched,"10,000",10000,"79.8% [78.4, 81.1]","0.5% [0.3, 0.7]"
AFR,Cross-population,Panel size,1000,"42.1% [41.1, 43.3]","1.7% [1.2, 2.3]"
AFR,Cross-population,893,893,"42.1% [41.0, 43.3]","1.7% [1.1, 2.3]"
AFR,Cross-population,"10,000",10000,"42.2% [41.1, 43.3]","2.4% [1.7, 3.2]"
AFR,Identity,893,893,"39.1% [37.8, 40.5]","67.6% [66.4, 68.7]"
AFR,Identity,"10,000",10000,"41.1% [39.7, 42.6]","70.0% [69.0, 70.9]"
EUR,Matched,Panel size,1000,"83.0% [81.7, 84.3]","2.2% [1.8, 2.7]"


## LD similarity: other population and identity vs the study LD

Per region; `pairs = "study_r_gt_0.05"` uses pairs with |r| > 0.05 in the study LD (as Fig 3B), `pairs = "all"` all upper-triangle pairs. The correlation with identity is undefined (`NA`). MSE against identity over all pairs is the mean off-diagonal r² of the study LD: smaller means closer to an identity matrix.

In [13]:
sim <- list.files(file.path(root, "ld_similarity"), pattern = "^ld_similarity_chunk_.*\\.rds$", full.names = TRUE) %>%
  map_dfr(readRDS) %>%
  mutate(reference_type = if_else(reference == "Identity", "Identity", "Other population"))

sim %>% group_by(study, reference, pairs) %>%
  summarise(regions = n(), across(c(mse, max_abs_diff, concord), ~ signif(median(.x, na.rm = TRUE), 3)), .groups = "drop") %>%
  as.data.frame()

## identity-likeness: mean off-diagonal r^2 of the AFR and EUR panels (MSE vs identity, all pairs), per region
sim %>% filter(reference == "Identity", pairs == "all") %>%
  select(region, study, mse) %>% pivot_wider(names_from = study, values_from = mse) %>%
  summarise(regions = n(), median_AFR = median(AFR), median_EUR = median(EUR), AFR_smaller = mean(AFR < EUR))

save_data(sim, paste0(data_name, "_LD_Similarity"))

study,reference,pairs,regions,mse,max_abs_diff,concord
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>
AFR,EUR,all,496,0.0176,1.21,0.633
AFR,EUR,study_r_gt_0.05,496,0.0325,1.21,0.690
AFR,Identity,all,496,0.0158,1.00,NA
AFR,Identity,study_r_gt_0.05,496,0.0377,1.00,NA
EUR,AFR,all,496,0.0176,1.21,0.633
EUR,AFR,study_r_gt_0.05,496,0.0378,1.21,0.681
EUR,Identity,all,496,0.0306,1.00,NA
EUR,Identity,study_r_gt_0.05,496,0.0735,1.00,NA


regions,median_AFR,median_EUR,AFR_smaller
<int>,<dbl>,<dbl>,<dbl>
496,0.01576484,0.03062093,1
